In [4]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import mne

# =========================
# CONFIG (single run only)
# =========================
BIDS_ROOT = Path(r"D:\ds003498")
SUBJECT = "01"
SESSION = "interictalsleep"
RUN = "01"

DERIV_NAME = "preproc_ds003498"
OUT_DIR = BIDS_ROOT / "derivatives" / DERIV_NAME / f"sub-{SUBJECT}" / f"ses-{SESSION}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Preproc choices
REF_MODE = "average"  # CAR
NOTCH_MAX_HZ = 500.0

# HFO bands (your later stage)
RIPPLE_BAND = (80.0, 250.0)
FAST_BAND = (250.0, 500.0)

# Skip notches too close to these edges (avoid nasty boundary effects)
EDGE_GUARD_HZ = 2.0
SKIP_NEAR_HZ = [RIPPLE_BAND[1], FAST_BAND[1]]  # 250, 500 by default

# QC plotting
QC_CH = "IAR1"      # set None to auto-pick first channel
QC_TMIN = 10.0
QC_TMAX = 12.0
QC_PSD_FMAX = 600.0

# =========================
# Helpers
# =========================
def find_vhdr(bids_root: Path, subject: str, session: str, run: str) -> Path:
    p = bids_root / f"sub-{subject}" / f"ses-{session}" / "ieeg" / f"sub-{subject}_ses-{session}_run-{run}_ieeg.vhdr"
    if not p.exists():
        raise FileNotFoundError(f"Missing vhdr: {p}")
    return p

def pick_qc_channel(raw: mne.io.BaseRaw, name: str | None) -> str:
    if name and name in raw.ch_names:
        return name
    return raw.ch_names[0]

def compute_psd_1ch(raw: mne.io.BaseRaw, ch_name: str, fmax: float) -> tuple[np.ndarray, np.ndarray]:
    psd = raw.compute_psd(method="welch", fmin=0.0, fmax=fmax, picks=[ch_name], verbose=False)
    return psd.freqs, psd.get_data().squeeze()

def mains_detect(raw: mne.io.BaseRaw, ch_name: str) -> int:
    """
    Decide 50 vs 60 by comparing PSD power near 50 and near 60.
    Uses a small window around each candidate and takes the max.
    """
    freqs, p = compute_psd_1ch(raw, ch_name, fmax=120.0)

    def peak_near(target: float, win_hz: float = 1.0) -> tuple[float, float]:
        m = (freqs >= target - win_hz) & (freqs <= target + win_hz)
        if not np.any(m):
            return np.nan, -np.inf
        idx = np.argmax(p[m])
        f_peak = freqs[m][idx]
        power = p[m][idx]
        return float(f_peak), float(power)

    f50, pow50 = peak_near(50.0, win_hz=1.0)
    f60, pow60 = peak_near(60.0, win_hz=1.0)

    print("Mains detection:")
    print(f"  candidate 50Hz: peak @ {f50:.6f} Hz | power={pow50:.3e}")
    print(f"  candidate 60Hz: peak @ {f60:.6f} Hz | power={pow60:.3e}")

    mains = 50 if pow50 >= pow60 else 60
    print(f"Chosen mains: {mains} Hz")
    return mains

def build_harmonics(mains: int, max_hz: float, sfreq: float, skip_near: list[float], guard_hz: float) -> list[float]:
    nyq = sfreq / 2.0
    hi = min(max_hz, nyq - 1.0)
    harms = []
    k = 1
    while mains * k <= hi:
        f = float(mains * k)
        # Skip near edges like 250/500 to avoid band-edge weirdness
        if any(abs(f - edge) <= guard_hz for edge in skip_near):
            k += 1
            continue
        harms.append(f)
        k += 1
    return harms

def notch_spectrum_fit(raw: mne.io.BaseRaw, freqs: list[float]) -> mne.io.BaseRaw:
    if not freqs:
        return raw.copy()
    r = raw.copy()
    # spectrum_fit supports multiple freqs in one call (unlike IIR)
    r.notch_filter(
        freqs=freqs,
        method="spectrum_fit",
        mt_bandwidth=2.0,  # tighter fit around each notch (tweakable)
        p_value=0.05,
        verbose=False,
    )
    return r

def plot_qc(raw0, raw1, raw2, ch: str, out_dir: Path, prefix: str):
    # PSD (in dB)
    f0, p0 = compute_psd_1ch(raw0, ch, QC_PSD_FMAX)
    f1, p1 = compute_psd_1ch(raw1, ch, QC_PSD_FMAX)
    f2, p2 = compute_psd_1ch(raw2, ch, QC_PSD_FMAX)

    plt.figure()
    plt.plot(f0, 10*np.log10(p0), label="raw")
    plt.plot(f1, 10*np.log10(p1), label="notch (spectrum_fit)")
    plt.plot(f2, 10*np.log10(p2), label="notch + CAR")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("PSD (dB)")
    plt.title(f"QC PSD | {prefix} | ch={ch}")
    plt.legend()
    psd_out = out_dir / f"{prefix}_qc_psd.png"
    plt.tight_layout()
    plt.savefig(psd_out, dpi=150)
    plt.close()

    # Time snippet
    sf = raw0.info["sfreq"]
    i0 = int(max(0, QC_TMIN * sf))
    i1 = int(min(raw0.n_times, QC_TMAX * sf))
    t = raw0.times[i0:i1]
    x0 = raw0.get_data(picks=[ch])[:, i0:i1].squeeze()
    x1 = raw1.get_data(picks=[ch])[:, i0:i1].squeeze()
    x2 = raw2.get_data(picks=[ch])[:, i0:i1].squeeze()

    plt.figure()
    plt.plot(t, x0, label="raw")
    plt.plot(t, x1, label="notch (spectrum_fit)")
    plt.plot(t, x2, label="notch + CAR")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.title(f"QC time series | ch={ch} | {QC_TMIN:.1f}-{QC_TMAX:.1f}s")
    plt.legend()
    ts_out = out_dir / f"{prefix}_qc_timeseries.png"
    plt.tight_layout()
    plt.savefig(ts_out, dpi=150)
    plt.close()

    print(f"[QC] Saved: {psd_out.name}, {ts_out.name}")

def plot_hfo_band_psd(raw: mne.io.BaseRaw, ch: str, out_dir: Path, prefix: str):
    """Optional: sanity-check PSD inside ripple/fast bands AFTER your preprocessing."""
    rr = raw.copy().filter(RIPPLE_BAND[0], RIPPLE_BAND[1], phase="zero", verbose=False)
    ff = raw.copy().filter(FAST_BAND[0], FAST_BAND[1], phase="zero", verbose=False)

    fr, pr = compute_psd_1ch(rr, ch, fmax=QC_PSD_FMAX)
    ffq, pf = compute_psd_1ch(ff, ch, fmax=QC_PSD_FMAX)

    plt.figure()
    plt.plot(fr, 10*np.log10(pr), label=f"ripple {RIPPLE_BAND[0]}-{RIPPLE_BAND[1]} Hz")
    plt.plot(ffq, 10*np.log10(pf), label=f"fast {FAST_BAND[0]}-{FAST_BAND[1]} Hz")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("PSD (dB)")
    plt.title(f"HFO-band PSD | {prefix} | ch={ch}")
    plt.legend()
    out = out_dir / f"{prefix}_qc_hfo_band_psd.png"
    plt.tight_layout()
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"[QC] Saved: {out.name}")

# =========================
# Main (single run)
# =========================
vhdr = find_vhdr(BIDS_ROOT, SUBJECT, SESSION, RUN)
raw = mne.io.read_raw_brainvision(str(vhdr), preload=True, verbose=False)
print(f"Loaded: sfreq={raw.info['sfreq']} Hz | ch={len(raw.ch_names)} | dur={raw.times[-1]:.1f}s")

qc_ch = pick_qc_channel(raw, QC_CH)

mains = mains_detect(raw, qc_ch)
notch_freqs = build_harmonics(
    mains=mains,
    max_hz=NOTCH_MAX_HZ,
    sfreq=raw.info["sfreq"],
    skip_near=SKIP_NEAR_HZ,
    guard_hz=EDGE_GUARD_HZ,
)
print(f"Notching harmonics up to {NOTCH_MAX_HZ} Hz (skipping near {SKIP_NEAR_HZ} ±{EDGE_GUARD_HZ}Hz):")
print(" ", notch_freqs)

raw_notch = notch_spectrum_fit(raw, notch_freqs)

raw_notch_car = raw_notch.copy()
raw_notch_car.set_eeg_reference(ref_channels=REF_MODE, verbose=False)

prefix = f"sub-{SUBJECT}_ses-{SESSION}_run-{RUN}"
plot_qc(raw, raw_notch, raw_notch_car, qc_ch, OUT_DIR, prefix)

# Optional but useful: check ripple/fast PSD after preprocessing
plot_hfo_band_psd(raw_notch_car, qc_ch, OUT_DIR, prefix)

C:\Users\ajars\AppData\Local\Temp\ipykernel_18084\184507283.py:180: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_brainvision(str(vhdr), preload=True, verbose=False)


Loaded: sfreq=2000.0 Hz | ch=50 | dur=300.0s
Mains detection:
  candidate 50Hz: peak @ 49.804688 Hz | power=1.972e-12
  candidate 60Hz: peak @ 60.546875 Hz | power=4.173e-13
Chosen mains: 50 Hz
Notching harmonics up to 500.0 Hz (skipping near [250.0, 500.0] ±2.0Hz):
  [50.0, 100.0, 150.0, 200.0, 300.0, 350.0, 400.0, 450.0]
[QC] Saved: sub-01_ses-interictalsleep_run-01_qc_psd.png, sub-01_ses-interictalsleep_run-01_qc_timeseries.png
[QC] Saved: sub-01_ses-interictalsleep_run-01_qc_hfo_band_psd.png
